# This is just for understanding the use of Pipelines

In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, GridSearchCV, KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_log_error, r2_score

In [2]:
# settintg the RandomState value 
Random = 42

# Downtown location instead of using top 1 percent expensive housing mean loaction as it was getting leaked 
Downtown_lat, Downtown_long = 47.6062, -122.3321         # Fixed point so no target lekage 

In [3]:
# i won't be doing any data analysis as i already knew the data 
df = pd.read_csv('kc_house_data.csv', low_memory= False , parse_dates= ['date'])
df = df.drop(['id','sqft_living15','sqft_lot15'], axis = 1)

# dropping the outlier in number of bedrooms 
df = df[df['bedrooms'] <15]

In [4]:
def haversine_km(lat1, long1, lat2, long2):

    lat1, long1, lat2, long2 = map(
        np.radians,
        [lat1, long1, lat2, long2]
    )

    a = (
        np.sin((lat2-lat1)/2)**2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin((long2-long1)/2)**2
    )

    return 6371 * 2 * np.arcsin(np.sqrt(a))

In [5]:
class HouseFeature(BaseEstimator, TransformerMixin):
    def fit(self, X, y = None):
        return self
    def transform(self, X):
        X = X.copy()
        X['sale_year'] = X.date.dt.year
        X['sale_month'] = X.date.dt.month
        X['age'] = (X['sale_year'] - X['yr_built']).clip(lower = 0)  # clip will replace all the value below 0 to 0
        X['is_renovated'] = (X['yr_renovated'] > 0).astype(int)
        X['has_basement'] = (X['sqft_basement'] > 0).astype(int)
        X["living_per_bedroom"] = X["sqft_living"] / X["bedrooms"].replace(0, 1)
        X["bath_per_bedroom"] = X["bathrooms"] / X["bedrooms"].replace(0, 1)
        X["living_per_lot"] = X["sqft_living"] / X["sqft_lot"]
        X["dist_downtown_km"] = haversine_km(X["lat"], X["long"], Downtown_lat, Downtown_long)
        return X.drop(columns=['date', 'yr_built', 'yr_renovated'])

In [6]:
# Creating features and label 
X, y = df.drop('price', axis = 1), df['price']

# spliting data into test and train set
X_train, X_test, y_train , y_test = train_test_split(X, y, test_size= 0.2, random_state= Random)

In [7]:
# creating list for the trasformation 
log_cols = ['sqft_living', 'sqft_lot', 'sqft_above', 'living_per_bedroom','living_per_lot']
scale_cols = ["bedrooms", "bathrooms", "floors", "view", "condition", "grade", "sqft_basement",
              "lat", "long", "age", "dist_downtown_km", "bath_per_bedroom", "sale_month"]
pass_cols = ['waterfront', 'is_renovated', 'has_basement', 'sale_year', 'zipcode']

In [8]:
def make_preprocessor(scale = True):
    num_tf = StandardScaler() if scale else 'passthrough'
    log_tf = Pipeline(
        [('log', FunctionTransformer(np.log1p, feature_names_out = 'one-to-one'))]
        + ([('scale', StandardScaler())]if scale else[])
    )
    return ColumnTransformer([("log", log_tf, log_cols),
                              ("num", num_tf, scale_cols),
                              ("pass", "passthrough", pass_cols)])

In [9]:
def build_pipeline(model , scale = True):
    pipe = Pipeline([('features' , HouseFeature()),
                     ('prep', make_preprocessor(scale)),
                    ('model', model)])
    return TransformedTargetRegressor(regressor = pipe, func = np.log1p, inverse_func= np.expm1)

In [10]:
candidates = {
    'Ridge' : build_pipeline(Ridge(alpha = .05), scale = True),
    'RandomForest' : build_pipeline(RandomForestRegressor(n_estimators= 200, n_jobs = -1, random_state= Random), scale = False),
    'HistGB' : build_pipeline(HistGradientBoostingRegressor(random_state = Random), scale = False)
                              }

In [11]:
cv = KFold(10, shuffle=True, random_state= Random)
for name, est in candidates.items():
    s = cross_val_score(est, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error", n_jobs=1)
    print(f"{name:13s} CV MAE = {-s.mean():,.0f} (+/- {s.std():,.0f})")

Ridge         CV MAE = 95,358 (+/- 2,806)
RandomForest  CV MAE = 69,115 (+/- 3,625)
HistGB        CV MAE = 64,729 (+/- 2,539)


In [12]:
def show_score(model):
    test_preds = model.predict(X_test)
    scores = {
        'MAE' : mean_absolute_error(y_test, test_preds),
        'Test R^2' : r2_score(y_test, test_preds)
    }
    return scores 

In [13]:
# Predicting for HistGradientBoostingRegressor

In [14]:
param_dict = {
    "regressor__model__learning_rate": [0.03, 0.05, 0.08, 0.1],
    "regressor__model__max_iter": [200, 300, 500],
    "regressor__model__max_leaf_nodes": [15, 31, 50],
    "regressor__model__min_samples_leaf": [10, 20, 30],
    "regressor__model__l2_regularization": [0, 0.1, 1.0]
}
search = RandomizedSearchCV(candidates["HistGB"], param_dict, n_iter=12, cv=3,
                            scoring="neg_mean_absolute_error", random_state=Random, n_jobs=-1)

In [15]:
search.fit(X_train, y_train);

In [16]:
search.best_params_

{'regressor__model__min_samples_leaf': 30,
 'regressor__model__max_leaf_nodes': 31,
 'regressor__model__max_iter': 500,
 'regressor__model__learning_rate': 0.05,
 'regressor__model__l2_regularization': 1.0}

In [17]:
show_score(search)

{'MAE': 64414.563451309004, 'Test R^2': 0.9036623384621107}